<a href="https://colab.research.google.com/github/joryhh/Capstone-Project--Building-Agentic-AI-Systems/blob/main/Capstone_Main_Workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — Requirements Engineering Copilot

**Before running:** add `GROQ_API_KEY` and `LANGCHAIN_API_KEY` to Colab Secrets
(🔑 in the left sidebar) and switch **Notebook access** on for both.

In [ ]:
# Chroma is not used (an in-process InMemoryVectorStore is used instead), so it is
# removed along with its OpenTelemetry deps.
# numpy is deliberately NOT pinned: Colab ships numpy 2.x and its preinstalled packages
# are compiled against it. Downgrading to <2 leaves a mixed install and raises
# "ValueError: numpy.dtype size changed" on the first numpy-heavy import.
!pip uninstall -y chromadb opentelemetry-api opentelemetry-sdk \
    opentelemetry-exporter-otlp opentelemetry-exporter-otlp-proto-grpc -q

!pip install -qU langchain langchain-groq langgraph langgraph-supervisor pydantic python-docx \
    langchain-huggingface sentence-transformers langchain-text-splitters

import os
os.environ["CHROMA_SERVER_NO_TELEMETRY"] = "1"

In [ ]:
# CELL 0b — API keys (Colab Secrets, never hardcoded)
from google.colab import userdata

# Required: free key at https://console.groq.com/keys
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Required for §8 (LangSmith) — note the EXACT variable name.
# LANGSMITH_TRACING_V2 is not real and fails silently — this is LANGCHAIN_TRACING_V2.
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "capstone-requirements-reviewer"

In [ ]:
# CELL 1 — Shared schemas (freeze this with Member B before anyone else builds on it)
from __future__ import annotations
from typing import Literal, Optional
from pydantic import BaseModel, Field


class Requirement(BaseModel):
    """A single parsed requirement, as produced by B's Requirements Parser Tool."""
    id: str = Field(description="Stable identifier, e.g. 'REQ-001'")
    text: str = Field(description="The requirement statement, verbatim or lightly normalized")
    category: Optional[str] = Field(
        default=None,
        description="Optional grouping, e.g. 'authentication', 'notifications'",
    )


class ReviewFinding(BaseModel):
    """One issue raised by any reviewer, against one requirement."""
    requirement_id: str = Field(description="ID of the requirement this finding refers to")
    issue_type: Literal["missing", "ambiguous", "conflicting", "untestable"] = Field(
        description=(
            "missing: a requirement that should exist but doesn't; "
            "ambiguous: vague/subjective wording; "
            "conflicting: contradicts another requirement; "
            "untestable: cannot be objectively verified as met"
        )
    )
    severity: Literal["low", "medium", "high", "critical"] = Field(
        description="How much this issue would hurt the project if left unresolved"
    )
    reason: str = Field(description="One or two sentences explaining the issue")
    suggested_change: str = Field(description="A concrete, measurable rewrite or addition")


class ApprovedChange(BaseModel):
    """A PO decision on one ReviewFinding, as emitted by D's HITL and consumed by E's Editor."""
    requirement_id: str
    action: Literal["replace", "add", "apply_po_edit", "leave_unchanged"] = Field(
        description=(
            "replace: swap the requirement text with suggested_change; "
            "add: this is a brand-new requirement being inserted; "
            "apply_po_edit: use the PO's own edited_text instead of the suggestion; "
            "leave_unchanged: reviewer's finding was rejected, requirement stays as-is"
        )
    )
    edited_text: Optional[str] = Field(
        default=None, description="PO's own wording, only set when action == 'apply_po_edit'"
    )
    notes: Optional[str] = Field(default=None, description="Free-text PO rationale, optional")


print("Schemas frozen:", Requirement.__name__, ReviewFinding.__name__, ApprovedChange.__name__)

Schemas frozen: Requirement ReviewFinding ApprovedChange


In [ ]:
# CELL 1b — Requirements Parser Tool (Member B)
# Turns raw requirements text (pasted or from an uploaded file) into a list of
# structured Requirement items with stable IDs. This is a real tool: it reads
# document_text and extracts from it — it does not ignore its argument.
from langchain_core.tools import tool
from langchain_groq import ChatGroq
parser_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)


class RequirementDraft(BaseModel):
    """One extracted requirement, before an ID is assigned."""
    text: str = Field(description="The requirement statement, lightly normalized")
    category: Optional[str] = Field(
        default=None, description="Optional grouping, e.g. 'authentication', 'notifications'"
    )


class ParsedRequirementsDraft(BaseModel):
    """Container so the LLM can return a variable-length list via structured output."""
    requirements: list[RequirementDraft] = Field(
        description="Every distinct requirement statement found in the document, "
        "in the order they appear"
    )


structured_parser_llm = parser_llm.with_structured_output(ParsedRequirementsDraft)


@tool
def parse_requirements(document_text: str) -> list[dict]:
    """Parse a raw requirements document (pasted text or file contents) into
    structured Requirement items with stable IDs.

    Use this whenever you have unstructured requirements text and need it
    converted into individually addressable records for review.
    """
    if not document_text or not document_text.strip():
        return []

    prompt = f"""Extract every distinct requirement statement from the document below.
A requirement is a single sentence describing something the system shall/should/must do.
Split compound sentences into separate requirements if they describe separate behaviors.
Ignore headings, examples, and non-requirement prose.

DOCUMENT:
{document_text}
"""
    draft = structured_parser_llm.invoke(prompt)

    requirements = [
        Requirement(id=f"REQ-{i+1:03d}", text=d.text.strip(), category=d.category)
        for i, d in enumerate(draft.requirements)
    ]
    print(f"  [parser] extracted {len(requirements)} requirement(s) from {len(document_text)} chars")
    return [r.model_dump() for r in requirements]


def load_document_text(file_path: str) -> str:
    """Read raw text from an uploaded .txt or .docx file, for parse_requirements."""
    if file_path.endswith(".docx"):
        from docx import Document
        doc = Document(file_path)
        return "\n".join(p.text for p in doc.paragraphs)
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()

In [ ]:
test_doc = """
The system shall allow a student to log in with their university ID.
The system shall respond to search queries quickly.
Users must be able to reset their password via email.
"""

result = parse_requirements.invoke({"document_text": test_doc})
for r in result:
    print(r)

AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

In [ ]:
# CELL 2 — STUB reviewers. Replace each with B/C's real with_structured_output call
# before the final submission. These intentionally ignore most of their input;
# that is only acceptable because they are scaffolding, never submitted as-is.
from langgraph.func import task


@task
def stub_completeness_reviewer(project_description: str, requirements: list[Requirement]) -> list[ReviewFinding]:
    """STUB — real version compares project_description against requirements to find gaps."""
    print(f"  [stub] completeness_reviewer called with {len(requirements)} requirement(s)")
    return [
        ReviewFinding(
            requirement_id="REQ-000",
            issue_type="missing",
            severity="high",
            reason="[STUB] Placeholder gap — replace with B's real completeness check.",
            suggested_change="[STUB] Add the missing requirement B's reviewer will identify.",
        )
    ]


@task
def stub_ambiguity_reviewer(requirements: list[Requirement]) -> list[ReviewFinding]:
    """STUB — real version flags vague/subjective terms and proposes a measurable rewrite."""
    print(f"  [stub] ambiguity_reviewer called with {len(requirements)} requirement(s)")
    if not requirements:
        return []
    return [
        ReviewFinding(
            requirement_id=requirements[0].id,
            issue_type="ambiguous",
            severity="medium",
            reason="[STUB] Placeholder vague-term flag — replace with B's real ambiguity check.",
            suggested_change="[STUB] Rewrite with a measurable threshold.",
        )
    ]


@task
def stub_conflict_reviewer(requirements: list[Requirement]) -> list[ReviewFinding]:
    """STUB — real version (C) compares requirement pairs for contradictions."""
    print(f"  [stub] conflict_reviewer called with {len(requirements)} requirement(s)")
    if len(requirements) < 2:
        return []
    return [
        ReviewFinding(
            requirement_id=requirements[1].id,
            issue_type="conflicting",
            severity="critical",
            reason="[STUB] Placeholder contradiction — replace with C's real conflict check.",
            suggested_change="[STUB] Reconcile with the conflicting requirement.",
        )
    ]


@task
def stub_testability_reviewer(requirements: list[Requirement]) -> list[ReviewFinding]:
    """STUB — real version (C) flags requirements with no objective pass/fail criterion."""
    print(f"  [stub] testability_reviewer called with {len(requirements)} requirement(s)")
    if not requirements:
        return []
    return [
        ReviewFinding(
            requirement_id=requirements[-1].id,
            issue_type="untestable",
            severity="medium",
            reason="[STUB] Placeholder untestable flag — replace with C's real testability check.",
            suggested_change="[STUB] Add a measurable acceptance criterion.",
        )
    ]

In [ ]:
# CELL 2b — Real Completeness & Ambiguity Reviewers (Member B)
# Replaces stub_completeness_reviewer and stub_ambiguity_reviewer.
from langchain_groq import ChatGroq

reviewer_llm_b = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)


# --- Completeness Reviewer -------------------------------------------------

class MissingRequirementDraft(BaseModel):
    """One capability described in the project but not covered by any requirement."""
    reason: str = Field(
        description="Why this capability is missing, referencing the project description directly"
    )
    suggested_change: str = Field(
        description="The exact requirement to add, written as 'The system shall ...'"
    )
    severity: Literal["low", "medium", "high", "critical"] = Field(
        description="How much leaving this gap unaddressed would hurt the project"
    )


class CompletenessFindings(BaseModel):
    gaps: list[MissingRequirementDraft] = Field(
        description="One entry per capability the project description mentions that has "
        "no corresponding requirement. Empty list if the requirements fully cover the description."
    )


completeness_llm = reviewer_llm_b.with_structured_output(CompletenessFindings)


@task
def completeness_reviewer(project_description: Optional[str], requirements: list[Requirement]) -> list[ReviewFinding]:
    """Real completeness check: compares the project description against the parsed
    requirements to find capabilities that are described but never turned into a requirement."""
    if not project_description:
        return []

    existing = "\n".join(f"- {r.id}: {r.text}" for r in requirements) or "(no requirements provided)"
    prompt = f"""You are reviewing a requirements document for completeness against its project description.

PROJECT DESCRIPTION:
{project_description}

EXISTING REQUIREMENTS:
{existing}

Find every capability mentioned or clearly implied in the project description that has
NO corresponding requirement in the list above. Do not flag anything already covered,
even if worded differently. If everything is covered, return an empty list.
"""
    result = completeness_llm.invoke(prompt)

    findings = [
        ReviewFinding(
            requirement_id=f"GAP-{i+1:03d}",
            issue_type="missing",
            severity=g.severity,
            reason=g.reason,
            suggested_change=g.suggested_change,
        )
        for i, g in enumerate(result.gaps)
    ]
    print(f"  [completeness] found {len(findings)} gap(s)")
    return findings


# --- Ambiguity Reviewer ------------------------------------------------------

class AmbiguityFindingDraft(BaseModel):
    """One requirement flagged for vague/subjective wording."""
    requirement_id: str = Field(description="ID of the flagged requirement, copied exactly from the input")
    reason: str = Field(description="Which term(s) are vague and why they can't be objectively verified")
    suggested_change: str = Field(
        description="The requirement rewritten with vague terms replaced by a measurable threshold"
    )
    severity: Literal["low", "medium", "high", "critical"] = Field(
        description="How much this ambiguity would hurt if left unresolved"
    )


class AmbiguityFindings(BaseModel):
    findings: list[AmbiguityFindingDraft] = Field(
        description="One entry per requirement containing vague/subjective wording "
        "(e.g. 'quickly', 'user-friendly', 'fast', 'easy', 'robust'). Empty list if none."
    )


ambiguity_llm = reviewer_llm_b.with_structured_output(AmbiguityFindings)


@task
def ambiguity_reviewer(requirements: list[Requirement]) -> list[ReviewFinding]:
    """Real ambiguity check: flags vague/subjective terms and proposes a measurable rewrite."""
    if not requirements:
        return []

    listed = "\n".join(f"- {r.id}: {r.text}" for r in requirements)
    prompt = f"""Review these requirements for vague or subjective wording that cannot be
objectively tested (e.g. "quickly", "user-friendly", "fast", "easy", "robust", "efficient").

REQUIREMENTS:
{listed}

For each requirement containing such wording, explain what's vague and rewrite it with a
concrete, measurable threshold. If a requirement is already specific and testable, do NOT
include it in your findings, even if you can't think of a way to make it "better." An empty
findings list is a valid and expected answer when no requirement is ambiguous.
"""
    result = ambiguity_llm.invoke(prompt)

    findings = [
        ReviewFinding(
            requirement_id=f.requirement_id,
            issue_type="ambiguous",
            severity=f.severity,
            reason=f.reason,
            suggested_change=f.suggested_change,
        )
        for f in result.findings
    ]
    print(f"  [ambiguity] found {len(findings)} vague requirement(s)")
    return findings

In [ ]:
from langgraph.func import entrypoint
from langgraph.checkpoint.memory import InMemorySaver

_test_checkpointer = InMemorySaver()


@entrypoint(checkpointer=_test_checkpointer)
def _test_b_reviewers(inputs: dict) -> dict:
    reqs = [Requirement(**r) for r in inputs["requirements"]]
    comp = completeness_reviewer(inputs["project_description"], reqs).result()
    amb = ambiguity_reviewer(reqs).result()
    return {
        "completeness": [f.model_dump() for f in comp],
        "ambiguity": [f.model_dump() for f in amb],
    }


test_reqs = [
    {"id": "REQ-001", "text": "The system shall respond to search queries quickly."},
    {"id": "REQ-002", "text": "Users must be able to reset their password via email."},
]
test_description = (
    "A course-registration system for university students. Students should be able "
    "to register for courses, drop courses, and receive notifications about seat "
    "availability."
)

test_result = _test_b_reviewers.invoke(
    {"project_description": test_description, "requirements": test_reqs},
    {"configurable": {"thread_id": "test-b-reviewers"}},
)

print("\n--- Completeness ---")
for f in test_result["completeness"]:
    print(f)

print("\n--- Ambiguity ---")
for f in test_result["ambiguity"]:
    print(f)

---
# Member C — RAG pipeline & Reviewers

In [ ]:
# CELL 2b-corpus — RAG knowledge base (Member C)
# Writes the requirements-engineering corpus to disk so the loader has real documents.
# These are our own written summaries of established RE guidance — not copied standard
# text — so the repo carries no licensing problem. Add more files here anytime.
import os, textwrap
os.makedirs("/content/rag_corpus", exist_ok=True)

CORPUS = {}

CORPUS["conflict_patterns.md"] = r"""
# Common Requirement Conflict Patterns (review reference)

A conflict exists when two requirements cannot both be satisfied by any single
implementation. Reviewers should test each candidate pair against these patterns.

## Pattern 1 — Permission versus prohibition
One requirement grants an actor an unrestricted ability while another forbids the same
actor the same ability under a condition that can occur. Signal words: "at any time",
"always", "never", "under no circumstances".
Example: "Users may edit their profile at any time" conflicts with "Users cannot edit
their profile after account verification", because a verified user falls under both.

## Pattern 2 — Contradictory quantitative limits
Two requirements state different values for the same measurable property under the same
conditions, such as two different response-time ceilings or two different retention
periods for the same record type.

## Pattern 3 — Incompatible ordering
Two requirements each demand that a different step occur first in the same workflow, for
example requiring payment before confirmation while also requiring confirmation before
payment.

## Pattern 4 — Mutually exclusive states
Two requirements demand that the same entity be in two states that cannot hold at once,
such as requiring that a record be permanently immutable while also requiring that it be
editable by an administrator.

## Pattern 5 — Deadline versus exception without precedence
A general deadline rule and a narrower exception rule both apply to the same action, and
neither states which takes precedence. This is a genuine conflict until an explicit
precedence rule is added.

## What is NOT a conflict
Two requirements that address different actors, different entities, or mutually exclusive
preconditions are not in conflict. A general rule followed by an exception that explicitly
names its precedence is not a conflict. Requirements at different levels of detail, where
one refines the other, are not in conflict.
"""

CORPUS["re_quality_criteria.md"] = r"""
# Requirements Quality Criteria (study extract)

A well-written requirement is expected to satisfy several quality characteristics.
These characteristics are drawn from established requirements-engineering guidance
and restated here in our own words for use as a review reference.

## Unambiguous
A requirement is unambiguous when it has exactly one possible interpretation.
Words such as "quickly", "fast", "user-friendly", "efficient", "robust", "flexible",
"as appropriate", "if necessary", "etc." and "and/or" introduce ambiguity because two
readers can reasonably disagree about what satisfies them. Replace these with a stated
value, a stated condition, or a named standard.

## Verifiable (testable)
A requirement is verifiable when a finite, cost-effective process exists to check that
the delivered system meets it. In practice this means the requirement must have an
objective pass/fail criterion. A requirement that cannot be verified by inspection,
analysis, demonstration, or test is not verifiable and should be rewritten.
Non-verifiable phrasing includes "the system shall be easy to use", "the system shall
work well", and "the system shall be secure" with no stated threshold or standard.

## Measurable performance
Performance requirements must state a numeric target, the unit of measurement, and the
conditions under which the target applies. The recommended form is:
"The system shall <action> within <number> <unit> under <stated load or condition>."
For example, a response-time requirement should state both the time limit and the
concurrent-user load at which that limit must hold.

## Complete
A requirement is complete when it states the triggering condition, the actor, the system
response, and the outcome, with no "to be determined" content remaining. A requirements
set is complete when it covers every capability described in the project scope, including
error handling and exception behaviour for each described capability.

## Consistent
A requirements set is consistent when no requirement contradicts another. Conflicts
commonly appear as a general permission paired with a narrower prohibition covering the
same actor and action, as two different values specified for the same quantity, or as two
requirements that impose incompatible ordering on the same operation.

## Singular
A requirement should state exactly one need. A sentence joining two behaviours with "and"
usually should be split into two requirements so that each can be verified separately.

## Feasible and traceable
A requirement must be achievable within known constraints, and must be traceable back to a
stated stakeholder need or scope item and forward to its verification method.
"""

CORPUS["sample_srs_registration.md"] = r"""
# Sample SRS Extract — University Course Registration (reference exemplar)

This extract is provided as a model of correctly written requirements for a course
registration domain. Reviewers may compare submitted requirements against these.

REQ-101 The system shall allow an authenticated student to register for a course section
when the section has at least one available seat and the student has no outstanding
registration hold.
Acceptance: Given a section with 1 seat free and a student with no hold, when the student
submits a registration request, then the seat count decreases by 1 and the student appears
on the section roster.

REQ-102 The system shall prevent registration for a course section whose enrolled count
equals its seat capacity, and shall return an explanatory message naming the full section.

REQ-103 The system shall allow a student to drop a course section up to and including the
published add/drop deadline for the current term.

REQ-104 The system shall reject any drop request submitted after the published add/drop
deadline and shall return an explanatory message stating the deadline date.

REQ-105 The system shall notify a waitlisted student within 5 minutes of a seat becoming
available in the section for which they are waitlisted.

REQ-106 The system shall allow an academic advisor to view the complete registration
history of any student assigned to that advisor.

REQ-107 The system shall allow an academic advisor to override a registration hold, and
shall record the advisor identity, timestamp, and stated reason for every override.

REQ-108 The system shall generate a weekly enrollment report listing current enrolled
count and seat capacity for every course section, and shall make it available to
department administrators each Monday by 06:00 local time.

REQ-109 The system shall complete a registration transaction within 3 seconds under a load
of 500 concurrent registration requests.

REQ-110 The system shall record an audit entry for every registration, drop, and override
action, containing actor identity, action type, target section, and timestamp.
"""

CORPUS["srs_writing_template.md"] = r"""
# Requirement Statement Template and House Style

## Mandatory sentence form
Every functional requirement in this organisation is written as:

  The system shall <observable behaviour> [when <trigger>] [within <measurable limit>].

The auxiliary verb "shall" denotes a binding requirement. "Should" denotes a
recommendation and must not be used for contractual requirements. "Will" denotes a
statement of fact about the environment, not a requirement on the system.

## Identifier convention
Each requirement carries a unique, stable identifier of the form REQ-NNN. Identifiers are
never reused after a requirement is deleted.

## Acceptance criteria
Every requirement is accompanied by at least one acceptance criterion stated in
Given / When / Then form, so that a tester can determine pass or fail without consulting
the author.

## Prohibited vague terms
The following terms are rejected in review unless immediately followed by a numeric
threshold or a named external standard: quickly, fast, slow, user-friendly, intuitive,
easy, simple, seamless, efficient, optimised, robust, reliable, scalable, secure,
appropriate, adequate, sufficient, minimal, maximal, state-of-the-art, modern.

## Performance requirement examples
Acceptable: The system shall return search results within 2 seconds for result sets of up
to 500 records, with 200 concurrent users.
Rejected: The system shall return search results quickly.

## Availability requirements
Availability is expressed as a percentage measured over a stated period, together with the
maximum permitted duration of a single unplanned outage.
Acceptable: The system shall maintain 99.5% availability measured monthly, with no single
unplanned outage exceeding 30 minutes.
Rejected: The system shall be highly available.
"""


for name, body in CORPUS.items():
    with open(f"/content/rag_corpus/{name}", "w", encoding="utf-8") as fh:
        fh.write(body)
    print(f"wrote /content/rag_corpus/{name}  ({len(body)} chars)")

In [ ]:
# CELL 2c — RAG pipeline (Member C)
# load -> split -> embed -> store -> retrieve, all five steps explicit.
# Uses InMemoryVectorStore (langchain-core) because Cell 0 deliberately uninstalls
# chromadb; this keeps the dependency surface minimal and adds no new vector-DB service.
# Packages for this cell (langchain-huggingface, sentence-transformers) install in Cell 0.

import os
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.tools import tool

CORPUS_DIR = "/content/rag_corpus"

# --- 1. LOAD -----------------------------------------------------------------
# Plain-python load: keeps langchain-community (being sunset) out of the dependency
# tree. Still a genuine load step - real files off disk into Document objects.
standards_docs = [
    Document(page_content=p.read_text(encoding="utf-8"), metadata={"source": str(p)})
    for p in sorted(Path(CORPUS_DIR).glob("**/*.md"))
]
assert standards_docs, f"No documents loaded from {CORPUS_DIR} — run the corpus cell first."
print(f"[1/5] loaded {len(standards_docs)} document(s):")
for d in standards_docs:
    print(f"      {Path(d.metadata['source']).name}  ({len(d.page_content)} chars)")

# --- 2. SPLIT ----------------------------------------------------------------
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n## ", "\n### ", "\n\n", "\n", " ", ""],
)
standards_chunks = splitter.split_documents(standards_docs)
print(f"[2/5] split into {len(standards_chunks)} chunks "
      f"(avg {sum(len(c.page_content) for c in standards_chunks)//len(standards_chunks)} chars)")

# --- 3. EMBED ----------------------------------------------------------------
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print(f"[3/5] embedding model loaded (dim={len(embeddings.embed_query('probe'))})")

# --- 4. STORE ----------------------------------------------------------------
standards_store = InMemoryVectorStore.from_documents(standards_chunks, embeddings)
print(f"[4/5] vector store built with {len(standards_chunks)} vectors")

# --- 5. RETRIEVE -------------------------------------------------------------
standards_retriever = standards_store.as_retriever(search_kwargs={"k": 4})
print("[5/5] retriever ready (k=4)")


def retrieve_standards(query: str, k: int = 4) -> str:
    """Deterministic 2-step retrieval helper used inside the reviewers."""
    hits = standards_store.similarity_search(query, k=k)
    if not hits:
        return "(no relevant standards retrieved)"
    return "\n\n---\n\n".join(
        f"[source: {Path(h.metadata['source']).name}]\n{h.page_content}" for h in hits
    )


@tool
def search_requirements_standards(query: str) -> str:
    """Search the requirements-engineering knowledge base for guidance relevant to a query.

    The knowledge base contains requirements quality criteria, the organisation's
    requirement-writing template and prohibited vague terms, a sample SRS exemplar,
    and catalogued requirement conflict patterns.

    Use this when you need to check how a requirement should be written, whether a term
    is disallowed, or what a correctly-written equivalent looks like.
    """
    return retrieve_standards(query, k=4)

In [ ]:
# CELL 2d — Real Conflict + Testability & Standards Reviewers (Member C)
# Replaces stub_conflict_reviewer and stub_testability_reviewer.
# Both use 2-Step RAG: retrieve grounding passages, then one structured-output call.
from typing import Literal, Optional
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langgraph.func import task

reviewer_llm_c = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)


# --- Conflict Reviewer -------------------------------------------------------

class ConflictDraft(BaseModel):
    """One contradiction found between two requirements."""
    requirement_id: str = Field(
        description="ID of the FIRST requirement in the conflicting pair, copied exactly from the input"
    )
    conflicting_with_id: str = Field(
        description="ID of the SECOND requirement in the conflicting pair, copied exactly from the input"
    )
    pattern: str = Field(
        description="Which conflict pattern from the retrieved reference this matches, "
                    "e.g. 'permission versus prohibition'"
    )
    reason: str = Field(
        description="One or two sentences explaining why these two requirements cannot both be "
                    "satisfied, naming both IDs"
    )
    suggested_change: str = Field(
        description="A concrete rewrite of one or both requirements that removes the contradiction"
    )
    severity: Literal["low", "medium", "high", "critical"] = Field(
        description="How damaging this contradiction would be if it reached implementation"
    )


class ConflictFindings(BaseModel):
    conflicts: list[ConflictDraft] = Field(
        description="One entry per genuinely contradictory pair. Empty list if the requirements "
                    "are mutually consistent."
    )


conflict_llm = reviewer_llm_c.with_structured_output(ConflictFindings)


@task
def conflict_reviewer(requirements: list[Requirement]) -> list[ReviewFinding]:
    """Real conflict check: compares requirement pairs for contradictions, grounded in the
    retrieved catalogue of conflict patterns (2-Step RAG)."""
    if len(requirements) < 2:
        return []

    # Step 1 — retrieve
    context = retrieve_standards(
        "requirement conflict patterns contradiction permission prohibition "
        "incompatible ordering mutually exclusive what is not a conflict",
        k=3,
    )

    # Step 2 — generate, grounded
    listed = "\n".join(f"- {r.id}: {r.text}" for r in requirements)
    prompt = f"""You are the Conflict Reviewer. Find pairs of requirements that contradict
each other, using the reference material below to decide what does and does not count.

REFERENCE MATERIAL (retrieved from the requirements knowledge base):
{context}

REQUIREMENTS UNDER REVIEW:
{listed}

Compare the requirements against each other. Report a conflict only when two requirements
cannot both be satisfied by any single implementation. Requirements about different actors,
different entities, or mutually exclusive preconditions are NOT conflicts. A general rule
with an exception that explicitly states its precedence is NOT a conflict.

Copy requirement IDs exactly as given. If nothing genuinely contradicts, return an empty
list — that is a valid and expected answer.
"""
    result = conflict_llm.invoke(prompt)

    findings = [
        ReviewFinding(
            requirement_id=c.requirement_id,
            issue_type="conflicting",
            severity=c.severity,
            # ReviewFinding is frozen with a single requirement_id, so the partner ID is
            # carried in `reason` rather than changing the shared schema.
            reason=f"Conflicts with {c.conflicting_with_id} [{c.pattern}]. {c.reason}",
            suggested_change=c.suggested_change,
        )
        for c in result.conflicts
    ]
    print(f"  [conflict] found {len(findings)} contradiction(s)")
    return findings


# --- Testability & Standards Reviewer ----------------------------------------

class TestabilityDraft(BaseModel):
    """One requirement that cannot be objectively verified or breaches house style."""
    requirement_id: str = Field(description="ID of the flagged requirement, copied exactly from the input")
    reason: str = Field(
        description="Why this requirement has no objective pass/fail criterion, or which "
                    "house-style rule from the retrieved reference it breaches"
    )
    suggested_change: str = Field(
        description="The requirement rewritten so a tester can determine pass/fail, following "
                    "the retrieved template's sentence form and including a measurable threshold"
    )
    severity: Literal["low", "medium", "high", "critical"] = Field(
        description="How much this would hurt verification if left unresolved"
    )


class TestabilityFindings(BaseModel):
    findings: list[TestabilityDraft] = Field(
        description="One entry per requirement that is not objectively verifiable or breaches "
                    "the documented standards. Empty list if all are testable and compliant."
    )


testability_llm = reviewer_llm_c.with_structured_output(TestabilityFindings)


@task
def testability_standards_reviewer(requirements: list[Requirement]) -> list[ReviewFinding]:
    """Real testability + standards check: flags requirements with no objective pass/fail
    criterion, grounded in retrieved requirements-engineering standards (2-Step RAG)."""
    if not requirements:
        return []

    # Step 1 — retrieve, using the actual requirement text as the query so the passages
    # are relevant to THIS document, not a fixed lookup.
    query = ("verifiable testable acceptance criteria measurable threshold pass fail "
             "prohibited vague terms requirement sentence form " +
             " ".join(r.text for r in requirements))
    context = retrieve_standards(query, k=4)

    # Step 2 — generate, grounded
    listed = "\n".join(f"- {r.id}: {r.text}" for r in requirements)
    prompt = f"""You are the Testability & Standards Reviewer. Judge each requirement against
the retrieved standards below, not against your own preferences.

RETRIEVED STANDARDS:
{context}

REQUIREMENTS UNDER REVIEW:
{listed}

Flag a requirement when EITHER:
(a) it has no objective pass/fail criterion — no tester could decide whether it is met; or
(b) it breaches a rule stated in the retrieved standards, such as using a prohibited vague
    term without a numeric threshold, or omitting the required sentence form.

When you flag one, cite the specific rule from the retrieved standards in your reason, and
rewrite it following the retrieved template. Do not flag a requirement that is already
specific and verifiable. An empty findings list is a valid and expected answer.
"""
    result = testability_llm.invoke(prompt)

    findings = [
        ReviewFinding(
            requirement_id=f.requirement_id,
            issue_type="untestable",
            severity=f.severity,
            reason=f.reason,
            suggested_change=f.suggested_change,
        )
        for f in result.findings
    ]
    print(f"  [testability] found {len(findings)} unverifiable requirement(s)")
    return findings

In [ ]:
# CELL 2e — §3 retrieval verification (Member C)
# The rubric's exact failure test: ask a question whose answer is VERBATIM in the corpus.
# If this returns "not found", the pipeline is broken no matter how correct the code looks.
probe = "What is the maximum permitted duration of a single unplanned outage?"
hits = standards_store.similarity_search(probe, k=3)

print(f"QUERY: {probe}\n")
for i, h in enumerate(hits, 1):
    print(f"--- hit {i} | {Path(h.metadata['source']).name} ---")
    print(h.page_content[:320].strip(), "\n")

joined = " ".join(h.page_content for h in hits)
assert "30 minutes" in joined, (
    "RETRIEVAL FAILED — the answer is verbatim in srs_writing_template.md but was not "
    "retrieved. Restart the runtime and run top to bottom before debugging anything else."
)
print("[PASS] Verbatim answer ('no single unplanned outage exceeding 30 minutes') retrieved.")

# Second probe: prove the agentic tool entry point works too.
print("\n--- agentic tool call ---")
print(search_requirements_standards.invoke(
    {"query": "is the word user-friendly allowed in a requirement"})[:400])

In [ ]:
# CELL 2f — Stub swap verification (Member C -> hand to Member A)
from langgraph.func import entrypoint
from langgraph.checkpoint.memory import InMemorySaver
# Proves C's real reviewers replace the stubs cleanly. After this passes, A edits
# CELL 5 and CELL 8 to call conflict_reviewer / testability_standards_reviewer.
demo_reqs = [
    Requirement(id="REQ-001", text="The system shall let a student register for a course."),
    Requirement(id="REQ-002", text="The system shall respond quickly to registration requests."),
    Requirement(id="REQ-003", text="The system shall always allow a student to drop a course within 24 hours of registering."),
    Requirement(id="REQ-004", text="The system shall never allow a student to drop a course after the add/drop deadline."),
]

@entrypoint(checkpointer=InMemorySaver())
def _test_c_reviewers(inputs: dict) -> dict:
    reqs = [Requirement(**r) for r in inputs["requirements"]]
    conf = conflict_reviewer(reqs).result()
    test = testability_standards_reviewer(reqs).result()
    return {"conflict": [f.model_dump() for f in conf],
            "testability": [f.model_dump() for f in test]}

out = _test_c_reviewers.invoke(
    {"requirements": [r.model_dump() for r in demo_reqs]},
    {"configurable": {"thread_id": "test-c-reviewers"}},
)

print("\n--- Conflict ---")
for f in out["conflict"]:
    print(f)
print("\n--- Testability & Standards ---")
for f in out["testability"]:
    print(f)

assert not any("[STUB]" in f["reason"] for f in out["conflict"] + out["testability"]), \
    "Stub output detected — the real reviewers did not run."
print("\n[PASS] No stub text. C's reviewers are live.")

### ⚠️ Member A — two-line swap needed in `CELL 5` and `CELL 8`

Once `2f` passes, replace the stub calls so the real reviewers run:

```python
    if decision.run_conflict:
        futures.append(conflict_reviewer(requirements))               # was stub_conflict_reviewer
    if decision.run_testability:
        futures.append(testability_standards_reviewer(requirements))  # was stub_testability_reviewer
```

Until then `CELL 6/7/8` reports still contain `[STUB]` text. A and B's cells are otherwise untouched.

In [ ]:
# CELL 3 — Router decision schema + Supervisor-as-router task
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)


class RouterDecision(BaseModel):
    """The Supervisor's routing decision — which reviewers should run on this input."""
    run_completeness: bool = Field(
        description="True only if there is a project description to compare requirements against"
    )
    run_ambiguity: bool = Field(
        description="True if at least one requirement exists to check for vague wording"
    )
    run_conflict: bool = Field(
        description="True only if there are at least two requirements that could contradict each other"
    )
    run_testability: bool = Field(
        description="True if at least one requirement exists to check for measurable acceptance criteria"
    )
    reason: str = Field(description="One short sentence justifying which reviewers were chosen and why")


router_llm = llm.with_structured_output(RouterDecision)


@task
def supervisor_route(project_description: Optional[str], requirements: list[Requirement]) -> RouterDecision:
    """Supervisor-as-router: an LLM call with constrained output decides who runs.

    This is the routing primitive from the course's structured-routing lesson,
    NOT a keyword rule. The LLM sees the actual counts/content and decides;
    we only apply a deterministic override afterward for cases that are not
    a judgment call but a logical impossibility (see _enforce_hard_guardrails).
    """
    prompt = f"""You are the Supervisor for a requirements-review pipeline.
Decide which reviewers should run on this input. Do not run a reviewer whose
job is logically impossible given the input (e.g. you cannot check completeness
against a project description that does not exist).

CRITICAL INSTRUCTION: For the boolean routing fields (run_completeness, run_ambiguity, run_conflict, run_testability), you MUST output strict JSON boolean values (true or false), NOT strings ("true" or "false").

Project description: {project_description if project_description else "(none provided)"}
Number of requirements: {len(requirements)}
Requirements:
{chr(10).join(f"- [{r.id}] {r.text}" for r in requirements) if requirements else "(none)"}
"""
    decision = router_llm.invoke(prompt)
    return _enforce_hard_guardrails(decision, project_description, requirements)


def _enforce_hard_guardrails(
    decision: RouterDecision, project_description: Optional[str], requirements: list[Requirement]
) -> RouterDecision:
    """Deterministic floor UNDER the LLM's decision, for cases that are not a judgment
    call at all: e.g. it is not possible to run completeness review with zero
    description, or conflict review with fewer than two requirements to compare.
    This does not replace the LLM's routing — it only prevents it from attempting
    something structurally impossible, exactly like the `Literal` type constrains
    the value space in the structured-routing lesson.
    """
    if not project_description:
        decision.run_completeness = False
    if len(requirements) < 2:
        decision.run_conflict = False
    if not requirements:
        decision.run_ambiguity = False
        decision.run_testability = False
    return decision

In [ ]:
# CELL 4 — Supervisor-as-synthesizer
_SEVERITY_RANK = {"critical": 0, "high": 1, "medium": 2, "low": 3}


@task
def supervisor_synthesize(finding_lists: list[list[ReviewFinding]]) -> list[ReviewFinding]:
    """Flatten all reviewers' outputs, dedupe by (requirement_id, issue_type) keeping the
    highest-severity duplicate, then sort by severity (critical first).
    """
    flattened: list[ReviewFinding] = [f for group in finding_lists for f in group]

    deduped: dict[tuple[str, str], ReviewFinding] = {}
    for finding in flattened:
        key = (finding.requirement_id, finding.issue_type)
        existing = deduped.get(key)
        if existing is None or _SEVERITY_RANK[finding.severity] < _SEVERITY_RANK[existing.severity]:
            deduped[key] = finding

    ordered = sorted(deduped.values(), key=lambda f: _SEVERITY_RANK[f.severity])
    return ordered


def render_report(findings: list[ReviewFinding]) -> str:
    """Human-readable rendering of the final, deduped, severity-ordered report."""
    if not findings:
        return "No findings — requirements document passed all checks that ran."
    lines = ["# Requirements Review Report", ""]
    for f in findings:
        lines.append(f"- **[{f.severity.upper()}] {f.issue_type}** — `{f.requirement_id}`")
        lines.append(f"  - Reason: {f.reason}")
        lines.append(f"  - Suggested change: {f.suggested_change}")
    return "\n".join(lines)

In [ ]:
# CELL 5 — Full entrypoint: Orchestrator-Worker pattern via the Functional API
from langgraph.func import entrypoint
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()  # shared infra; D owns the long-term Store separately


@entrypoint(checkpointer=checkpointer)
def review_pipeline(inputs: dict) -> dict:
    """Orchestrator-Worker requirements-review pipeline.

    inputs = {
        "project_description": str | None,
        "requirements": list[dict],  # each dict must match the Requirement schema
    }
    """
    project_description = inputs.get("project_description")
    requirements = [Requirement(**r) for r in inputs.get("requirements", [])]

    # --- Orchestrator: decide who runs -----------------------------------
    decision = supervisor_route(project_description, requirements).result()
    print(f"Routing decision: {decision.model_dump()}")

    # --- Workers: call ONLY the reviewers the orchestrator selected -------
    # Futures are collected first (not .result()'d immediately) so genuinely
    # selected reviewers can run concurrently — this is what makes it
    # Orchestrator-Worker rather than a serial chain.
    futures = []
    if decision.run_completeness:
        futures.append(completeness_reviewer(project_description, requirements))
    if decision.run_ambiguity:
        futures.append(ambiguity_reviewer(requirements))
    if decision.run_conflict:
        futures.append(stub_conflict_reviewer(requirements))
    if decision.run_testability:
        futures.append(stub_testability_reviewer(requirements))

    finding_lists = [f.result() for f in futures]

    # --- Synthesizer: merge, dedupe, order --------------------------------
    final_findings = supervisor_synthesize(finding_lists).result()

    return {
        "routing_decision": decision.model_dump(),
        "findings": [f.model_dump() for f in final_findings],
        "report": render_report(final_findings),
    }

In [ ]:
# CELL 6 — Run 1: full input (description + 3 requirements) — expect all 4 reviewers to run
config_a = {"configurable": {"thread_id": "demo-full-input"}}

full_input = {
    "project_description": (
        "A course-registration system for university students. Students should be able "
        "to register for courses, drop courses, and receive notifications about seat "
        "availability."
    ),
    "requirements": [
        {"id": "REQ-001", "text": "The system shall let a student register for a course."},
        {"id": "REQ-002", "text": "The system shall respond quickly to registration requests."},
        {"id": "REQ-003", "text": "The system shall never allow a student to drop a course after the add/drop deadline, "
                                   "except that the system shall always allow drops within 24 hours of registration."},
    ],
}

result_a = review_pipeline.invoke(full_input, config_a)
print("\n--- REPORT (full input) ---")
print(result_a["report"])

In [ ]:
# CELL 7 — Run 2: no description, 1 requirement — expect completeness AND conflict to be skipped
config_b = {"configurable": {"thread_id": "demo-sparse-input"}}

sparse_input = {
    "project_description": None,
    "requirements": [
        {"id": "REQ-010", "text": "The system shall be user-friendly."},
    ],
}

result_b = review_pipeline.invoke(sparse_input, config_b)
print("\n--- REPORT (sparse input) ---")
print(result_b["report"])

# Direct proof of selectivity: compare the two routing decisions
print("\nRun 1 decision:", result_a["routing_decision"])
print("Run 2 decision:", result_b["routing_decision"])
assert result_a["routing_decision"]["run_completeness"] is True
assert result_b["routing_decision"]["run_completeness"] is False
assert result_b["routing_decision"]["run_conflict"] is False
print("\n✅ Selectivity confirmed: routing genuinely differs by input.")

In [ ]:
# CELL 8 — Integration smoke test: routing + checkpointer + a real interrupt(),
# standing in for D's HITL node ahead of E's Editor. Replace `pending_editor_step`
# with D and E's real implementation once both land — this cell only proves the
# three pieces (routing, checkpointer, interrupt) coexist without breaking each other.
from langgraph.types import interrupt, Command


@task
def pending_editor_step(findings: list[ReviewFinding]) -> str:
    """STUB for D+E's real Approve/Edit/Reject flow. Pauses before any requirement
    text is actually changed — same placement the real Editor will use.
    """
    decision = interrupt({
        "action": "Approve, edit, or reject the proposed findings before they're applied",
        "findings": [f.model_dump() for f in findings],
    })
    return f"[STUB] Editor received PO decision: {decision}"


@entrypoint(checkpointer=checkpointer)
def review_pipeline_with_pause(inputs: dict) -> dict:
    project_description = inputs.get("project_description")
    requirements = [Requirement(**r) for r in inputs.get("requirements", [])]

    decision = supervisor_route(project_description, requirements).result()

    futures = []
    if decision.run_completeness:
        futures.append(completeness_reviewer(project_description, requirements))
    if decision.run_ambiguity:
        futures.append(ambiguity_reviewer(requirements))
    if decision.run_conflict:
        futures.append(stub_conflict_reviewer(requirements))
    if decision.run_testability:
        futures.append(stub_testability_reviewer(requirements))

    finding_lists = [f.result() for f in futures]
    final_findings = supervisor_synthesize(finding_lists).result()

    editor_result = pending_editor_step(final_findings).result()
    return {"routing_decision": decision.model_dump(), "editor_result": editor_result}


# --- Run it: pause, then resume ---
cfg = {"configurable": {"thread_id": "integration-check-1"}}

paused = review_pipeline_with_pause.invoke(full_input, cfg)
print("PAUSED — interrupt payload:", paused["__interrupt__"][0].value["action"])
print("Findings awaiting approval:", len(paused["__interrupt__"][0].value["findings"]))

resumed = review_pipeline_with_pause.invoke(Command(resume="approve"), cfg)
print("\nRESUMED — final result:", resumed)

In [ ]:
# Demo input written inline so the notebook is self-contained (Colab wipes /content on restart).
with open("/content/demo_requirements_flawed.txt", "w", encoding="utf-8") as f:
    f.write("""The system shall allow a student to browse the course catalogue.
The system shall let a student register for a course section.
The system shall respond quickly to registration requests.
The system shall be user-friendly for first-time students.
The system shall always allow a student to drop a course within 24 hours of registering.
The system shall never allow a student to drop a course after the add/drop deadline.
The system shall notify a waitlisted student when a seat becomes available.
The system shall be reliable and provide a good user experience.
The system shall allow an academic advisor to override a registration hold.
""")

In [ ]:
# test
doc_text = load_document_text("/content/demo_requirements_flawed.txt")
parsed = parse_requirements.invoke({"document_text": doc_text})

project_description = """BrightPath is a web-based course registration system for university students.
The platform allows students to browse available courses, register for courses,
and drop courses they are no longer able to attend. Because popular courses
often fill up quickly, students should be notified automatically when a seat
becomes available in a course they are waitlisted for. Academic advisors need
to be able to review a student's registration history and override registration
holds when necessary. The system must also generate a weekly enrollment report
for department administrators showing current seat counts per course."""

full_demo_input = {
    "project_description": project_description,
    "requirements": parsed,
}

cfg = {"configurable": {"thread_id": "full-demo-run"}}
final_result = review_pipeline.invoke(full_demo_input, cfg)

print(final_result["report"])

In [ ]:
from uuid import uuid4
from datetime import datetime, timezone
from typing import Literal, Optional
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langgraph.store.memory import InMemoryStore
from langgraph.func import task, entrypoint
from langgraph.types import interrupt, Command

project_memory_store = InMemoryStore()


def _project_memory_namespace(project_id: str):
    project_id = project_id.strip()
    if not project_id:
        raise ValueError("project_id must not be empty.")
    return (project_id, "po_decisions")


@tool
def save_project_memory(
    project_id: str,
    requirement_id: str,
    decision: str,
    final_text: str = "",
    notes: str = "",
) -> str:
    """Save a Product Owner decision to long-term project memory."""
    namespace = _project_memory_namespace(project_id)
    memory_id = str(uuid4())

    project_memory_store.put(
        namespace,
        memory_id,
        {
            "project_id": project_id,
            "requirement_id": requirement_id,
            "decision": decision,
            "final_text": final_text or None,
            "notes": notes or None,
            "saved_at": datetime.now(timezone.utc).isoformat(),
        },
    )

    return memory_id


@tool
def retrieve_project_memory(project_id: str) -> list[dict]:
    """Retrieve Product Owner decisions from long-term project memory."""
    namespace = _project_memory_namespace(project_id)
    items = project_memory_store.search(namespace, limit=100)

    return [
        {
            "memory_id": item.key,
            **item.value,
        }
        for item in items
    ]


@entrypoint(checkpointer=checkpointer)
def memory_write_test(inputs: dict) -> dict:
    memory_id = save_project_memory.invoke(
        {
            "project_id": inputs["project_id"],
            "requirement_id": inputs["requirement_id"],
            "decision": inputs["decision"],
            "final_text": inputs.get("final_text", ""),
            "notes": inputs.get("notes", ""),
        }
    )

    return {
        "memory_id": memory_id,
        "project_id": inputs["project_id"],
    }


@entrypoint(checkpointer=checkpointer)
def memory_read_test(inputs: dict) -> dict:
    memories = retrieve_project_memory.invoke(
        {
            "project_id": inputs["project_id"]
        }
    )

    return {
        "project_id": inputs["project_id"],
        "memories": memories,
    }


memory_test_project = "brightpath-memory-test"

memory_thread_a = {
    "configurable": {
        "thread_id": "memory-thread-A"
    }
}

memory_thread_b = {
    "configurable": {
        "thread_id": "memory-thread-B"
    }
}

memory_write_result = memory_write_test.invoke(
    {
        "project_id": memory_test_project,
        "requirement_id": "REQ-002",
        "decision": "approve",
        "final_text": "The system shall respond to registration requests within 2 seconds.",
        "notes": "Approved measurable response-time requirement.",
    },
    memory_thread_a,
)

memory_read_result = memory_read_test.invoke(
    {
        "project_id": memory_test_project
    },
    memory_thread_b,
)

assert memory_thread_a["configurable"]["thread_id"] != memory_thread_b["configurable"]["thread_id"]

assert any(
    item["requirement_id"] == "REQ-002"
    and item["decision"] == "approve"
    for item in memory_read_result["memories"]
)

print("CROSS-THREAD MEMORY PASS")

for item in memory_read_result["memories"]:
    print(item)


class POFindingDecision(BaseModel):
    finding_index: int = Field(ge=0)
    decision: Literal["approve", "edit", "reject"]
    edited_text: Optional[str] = None
    notes: Optional[str] = None


class POReviewResponse(BaseModel):
    decisions: list[POFindingDecision]


@task
def requirements_approval_hitl(
    project_id: str,
    findings: list[ReviewFinding],
    previous_project_decisions: list[dict],
) -> dict:

    response = interrupt(
        {
            "type": "requirements_review_approval",
            "project_id": project_id,
            "message": "Review every finding and choose approve, edit, or reject.",
            "previous_project_decisions": previous_project_decisions,
            "findings": [
                {
                    "finding_index": index,
                    **finding.model_dump(),
                    "allowed_decisions": [
                        "approve",
                        "edit",
                        "reject",
                    ],
                }
                for index, finding in enumerate(findings)
            ],
        }
    )

    return response


@task
def build_approved_changes(
    findings: list[ReviewFinding],
    po_response: dict,
) -> list[ApprovedChange]:

    response = POReviewResponse(**po_response)

    if len(response.decisions) != len(findings):
        raise ValueError(
            f"Expected {len(findings)} decisions, received {len(response.decisions)}."
        )

    decisions_by_index = {}

    for decision in response.decisions:
        if decision.finding_index in decisions_by_index:
            raise ValueError(
                f"Duplicate decision for finding {decision.finding_index}."
            )

        decisions_by_index[decision.finding_index] = decision

    expected_indexes = set(range(len(findings)))
    received_indexes = set(decisions_by_index.keys())

    if expected_indexes != received_indexes:
        raise ValueError(
            "Every finding must have exactly one Product Owner decision."
        )

    approved_changes = []

    for index, finding in enumerate(findings):
        decision = decisions_by_index[index]

        if decision.decision == "approve":
            if finding.issue_type == "missing":
                approved_changes.append(
                    ApprovedChange(
                        requirement_id=finding.requirement_id,
                        action="add",
                        edited_text=finding.suggested_change,
                        notes=decision.notes,
                    )
                )
            else:
                approved_changes.append(
                    ApprovedChange(
                        requirement_id=finding.requirement_id,
                        action="replace",
                        edited_text=finding.suggested_change,
                        notes=decision.notes,
                    )
                )

        elif decision.decision == "edit":
            if not decision.edited_text or not decision.edited_text.strip():
                raise ValueError(
                    f"Finding {index} requires edited_text."
                )

            approved_changes.append(
                ApprovedChange(
                    requirement_id=finding.requirement_id,
                    action="apply_po_edit",
                    edited_text=decision.edited_text.strip(),
                    notes=decision.notes,
                )
            )

        else:
            approved_changes.append(
                ApprovedChange(
                    requirement_id=finding.requirement_id,
                    action="leave_unchanged",
                    edited_text=None,
                    notes=decision.notes,
                )
            )

    return approved_changes


@task
def persist_po_decisions(
    project_id: str,
    findings: list[ReviewFinding],
    approved_changes: list[ApprovedChange],
) -> list[str]:

    if len(findings) != len(approved_changes):
        raise ValueError(
            "findings and approved_changes must contain the same number of items."
        )

    memory_ids = []

    for finding, change in zip(findings, approved_changes):

        if change.action == "leave_unchanged":
            decision = "reject"

        elif change.action == "apply_po_edit":
            decision = "edit"

        else:
            decision = "approve"

        memory_id = save_project_memory.invoke(
            {
                "project_id": project_id,
                "requirement_id": finding.requirement_id,
                "decision": decision,
                "final_text": change.edited_text or "",
                "notes": change.notes or "",
            }
        )

        memory_ids.append(memory_id)

    return memory_ids


@entrypoint(checkpointer=checkpointer)
def member_d_pipeline(inputs: dict) -> dict:

    project_id = inputs["project_id"]
    project_description_input = inputs.get("project_description")

    requirements = [
        Requirement(**item)
        for item in inputs.get("requirements", [])
    ]

    previous_project_decisions = retrieve_project_memory.invoke(
        {
            "project_id": project_id
        }
    )

    routing_decision = supervisor_route(
        project_description_input,
        requirements,
    ).result()

    futures = []

    if routing_decision.run_completeness:
        futures.append(
            completeness_reviewer(
                project_description_input,
                requirements,
            )
        )

    if routing_decision.run_ambiguity:
        futures.append(
            ambiguity_reviewer(requirements)
        )

    if routing_decision.run_conflict:
        futures.append(
            conflict_reviewer(requirements)
        )

    if routing_decision.run_testability:
        futures.append(
            testability_standards_reviewer(requirements)
        )

    finding_lists = [
        future.result()
        for future in futures
    ]

    final_findings = supervisor_synthesize(
        finding_lists
    ).result()

    human_response = requirements_approval_hitl(
        project_id,
        final_findings,
        previous_project_decisions,
    ).result()

    approved_changes = build_approved_changes(
        final_findings,
        human_response,
    ).result()

    saved_memory_ids = persist_po_decisions(
        project_id,
        final_findings,
        approved_changes,
    ).result()

    return {
        "project_id": project_id,
        "routing_decision": routing_decision.model_dump(),
        "previous_project_decisions": previous_project_decisions,
        "findings": [
            finding.model_dump()
            for finding in final_findings
        ],
        "report": render_report(final_findings),
        "approved_changes": [
            change.model_dump()
            for change in approved_changes
        ],
        "saved_memory_ids": saved_memory_ids,
    }


d_input = {
    "project_id": "brightpath-capstone",
    "project_description": project_description,
    "requirements": parsed,
}

d_config = {
    "configurable": {
        "thread_id": "member-d-hitl-main"
    }
}

paused_result = member_d_pipeline.invoke(
    d_input,
    d_config,
)

assert "__interrupt__" in paused_result

hitl_payload = paused_result["__interrupt__"][0].value

print("\nWORKFLOW PAUSED")
print(hitl_payload["message"])
print("Findings awaiting review:", len(hitl_payload["findings"]))

for item in hitl_payload["findings"]:
    print("\n--------------------------------")
    print("Finding index:", item["finding_index"])
    print("Requirement:", item["requirement_id"])
    print("Issue type:", item["issue_type"])
    print("Severity:", item["severity"])
    print("Reason:", item["reason"])
    print("Suggested change:", item["suggested_change"])


po_decisions = []

for item in hitl_payload["findings"]:

    while True:
        choice = input(
            f"\nFinding {item['finding_index']} - approve / edit / reject: "
        ).strip().lower()

        if choice in {"approve", "edit", "reject"}:
            break

        print("Please enter approve, edit, or reject.")

    edited_text = None

    if choice == "edit":
        edited_text = input(
            "Enter the edited requirement: "
        ).strip()

        while not edited_text:
            edited_text = input(
                "Edited requirement cannot be empty. Enter it again: "
            ).strip()

    notes_input = input(
        "Optional Product Owner notes. Press Enter to skip: "
    ).strip()

    po_decisions.append(
        {
            "finding_index": item["finding_index"],
            "decision": choice,
            "edited_text": edited_text,
            "notes": notes_input or None,
        }
    )


resume_payload = {
    "decisions": po_decisions
}

resumed_result = member_d_pipeline.invoke(
    Command(
        resume=resume_payload
    ),
    d_config,
)

assert "__interrupt__" not in resumed_result

assert len(resumed_result["approved_changes"]) == len(
    resumed_result["findings"]
)

print("\nWORKFLOW RESUMED")

print("\nAPPROVED CHANGES")

for change in resumed_result["approved_changes"]:
    print(change)

print("\nHITL RESUME PASS")


@entrypoint(checkpointer=checkpointer)
def new_thread_memory_test(inputs: dict) -> dict:

    memories = retrieve_project_memory.invoke(
        {
            "project_id": inputs["project_id"]
        }
    )

    return {
        "memories": memories
    }


new_thread_config = {
    "configurable": {
        "thread_id": "member-d-second-session"
    }
}

new_thread_result = new_thread_memory_test.invoke(
    {
        "project_id": "brightpath-capstone"
    },
    new_thread_config,
)

assert len(new_thread_result["memories"]) > 0

print("\nLONG-TERM MEMORY FROM NEW THREAD")

for item in new_thread_result["memories"]:
    print(item)

print("\nLONG-TERM MEMORY CROSS-THREAD PASS")
print("MEMBER D COMPLETE")

In [ ]:
# CELL 9 — Error Handling: Retry + Fallback
# Strategy 1: RetryPolicy on reviewer tasks (exponential backoff, transient errors only)
# Strategy 2: Fallback — structured "reviewer unavailable" finding instead of crashing

from langgraph.types import RetryPolicy

# --- Strategy 1: Retry ---
reviewer_retry_policy = RetryPolicy(
    max_attempts=3,
    initial_interval=1.0,
    backoff_factor=2.0,
    max_interval=10.0,
    jitter=True,
    retry_on=(TimeoutError, ConnectionError),
)

# --- Strategy 2: Fallback ---
def make_fallback_finding(requirement_id: str, error: Exception) -> ReviewFinding:
    return ReviewFinding(
        requirement_id=requirement_id,
        issue_type="untestable",  # closest valid type; reason explains the real cause
        severity="low",
        reason=f"Reviewer failed after all retries: {type(error).__name__}: {error}",
        suggested_change="Manual review required — automated reviewer was unavailable.",
    )


def with_fallback(reviewer_fn):
    """Wrap a reviewer so it returns a fallback finding instead of crashing the run."""
    def wrapped(requirement, *args, **kwargs) -> ReviewFinding:
        try:
            return reviewer_fn(requirement, *args, **kwargs)
        except Exception as e:
            return make_fallback_finding(requirement.id, e)
    return wrapped


# --- Smoke test ---
def _flaky_reviewer(requirement):
    raise ConnectionError("simulated LLM API outage")

class _FakeReq:
    id = "REQ-999"

_safe = with_fallback(_flaky_reviewer)
_result = _safe(_FakeReq())
print(f"§6 smoke test — fallback fired: {_result}")
assert _result.issue_type == "untestable"
assert "ConnectionError" in _result.reason
print("✅ Retry policy defined + fallback works")

In [ ]:
# CELL 10 — Requirements Editor Agent

class EditorError(Exception):
    """Raised when an ApprovedChange can't be applied."""


def _apply_one(by_id: dict, change: ApprovedChange, next_new_id: int) -> int:
    if change.action == "leave_unchanged":
        return next_new_id

    if change.action == "replace":
        if change.requirement_id not in by_id:
            raise EditorError(f"Cannot replace unknown requirement_id={change.requirement_id!r}")
        # For replace, use edited_text if provided, otherwise use the suggested_change
        # from the finding (passed through by D's HITL flow)
        new_text = change.edited_text
        if not new_text:
            raise EditorError(f"replace on {change.requirement_id!r} missing edited_text")
        by_id[change.requirement_id].text = new_text
        return next_new_id

    if change.action == "apply_po_edit":
        if change.requirement_id not in by_id:
            raise EditorError(f"Cannot apply PO edit to unknown requirement_id={change.requirement_id!r}")
        if not change.edited_text:
            raise EditorError(f"apply_po_edit on {change.requirement_id!r} missing edited_text")
        by_id[change.requirement_id].text = change.edited_text
        return next_new_id

    if change.action == "add":
        if not change.edited_text:
            raise EditorError("add action missing edited_text")
        new_id = change.requirement_id
        if not new_id or new_id in by_id:
            new_id = f"REQ-NEW-{next_new_id:03d}"
            next_new_id += 1
        by_id[new_id] = Requirement(id=new_id, text=change.edited_text)
        return next_new_id

    raise EditorError(f"Unknown action: {change.action!r}")


def _apply_approved_changes(
    requirements: list[Requirement],
    decisions: list[ApprovedChange],
) -> list[Requirement]:
    """Apply a batch of ApprovedChange decisions to a requirements list."""
    by_id = {r.id: r for r in requirements}
    next_new_id = 1
    for change in decisions:
        next_new_id = _apply_one(by_id, change, next_new_id)
    return list(by_id.values())


apply_approved_changes = task(_apply_approved_changes)


# --- Smoke test ---
_test_reqs = [
    Requirement(id="REQ-001", text="The system should respond quickly."),
    Requirement(id="REQ-002", text="Users can log in."),
]
_test_changes = [
    ApprovedChange(requirement_id="REQ-001", action="replace",
                   edited_text="The system shall respond within 200ms for 95% of requests."),
    ApprovedChange(requirement_id="REQ-002", action="leave_unchanged"),
    ApprovedChange(requirement_id="REQ-NEW", action="add",
                   edited_text="Users shall receive an email notification on course registration."),
]
_updated = _apply_approved_changes(_test_reqs, _test_changes)
for _r in _updated:
    print(_r.id, "->", _r.text)

assert _updated[0].text == "The system shall respond within 200ms for 95% of requests."
assert _updated[1].text == "Users can log in."  # leave_unchanged
assert len(_updated) == 3  # one added
print("\n✅ Editor agent works — replace, leave_unchanged, and add all verified")